In [1]:
# 核心思想：迭代从网络架构中找出网络组件，当网络中存在与父节点不一致的学习率时，将架构分开拆分
# 函数的接口定义：
# 输入：模型 -> 输出：params_list(包括了学习率), 是否存在不同的状态变量
# 递归的三要素：递归元素之间的相互逻辑、递归传递状态的维护、递归的终点
def get_params(model, lr_predecessor=None):
    params_list = []  # 参数与学习率列表
    status = False  # 指示在该模型的子模块内是否存在其它学习率的设置
    lr = lr_predecessor  # 前继节点的设置学习率

    if hasattr(model, 'lr') and model.lr is not None:  # 判断该节点是否存在学习率设置
        status = True
        lr = model.lr

    for child in model.children():
        params_list_child, status_child = get_params(child, lr)
        if not params_list_child:  # 判断child是否为最底层的节点(当child没有子节点时，其不进入循环，返回列表为空)
            if status_child:  # 判断底层节点是否有自定义的学习率
                params_list_child = [{'params': child.parameters(), 'lr': child.lr}]
            elif lr is not None:  # 判断父节点是否有传递的学习率
                params_list_child = [{'params': child.parameters(), 'lr': lr}]
            else:  # 无学习率要求
                params_list_child = [{'params': child.parameters()}]
        if status_child:  # 只要子网络组分中存在一个模型有学习率设置，则需要拆分网络模块表以添加该学习率
            status = True
        params_list.extend(params_list_child)

    if not status and lr is None:  # 表明该模型中没有找到不同的学习率组件且没有指定的前继学习率
        return [{'params': model.parameters()}], status

    return params_list, status

In [2]:
def get_params_v2(model, lr_predecessor=None):
    params_list = []  # 参数与学习率列表
    status = False  # 指示在该模型的子模块内是否存在其它学习率的设置
    update_mark = True  # 指示模型是否被添加到优化器
    lr = lr_predecessor  # 前继节点的设置学习率

    if hasattr(model, 'update_mark') and not model.update_mark:  # 判断该节点是否需要更新参数
        update_mark = False
        return [], status, update_mark

    if hasattr(model, 'lr') and model.lr is not None:  # 判断该节点是否存在学习率设置
        status = True
        lr = model.lr

    for child in model.children():
        params_list_child, status_child, update_mark_child = get_params_v2(child, lr)
        if not update_mark_child:  # 判断子节点是否需要更新参数，如果不需要，则跳过本回合
            continue
        if not params_list_child:  # 判断child是否为最底层的节点(当child没有子节点时，其不进入循环，返回列表为空)
            if status_child:  # 判断底层节点是否有自定义的学习率
                params_list_child = [{'params': child.parameters(), 'lr': child.lr}]
            elif lr is not None:  # 判断父节点是否有传递的学习率
                params_list_child = [{'params': child.parameters(), 'lr': lr}]
            else:  # 无学习率要求
                params_list_child = [{'params': child.parameters()}]
        if status_child:  # 只要子网络组分中存在一个模型有学习率设置，则需要拆分网络模块表以添加该学习率
            status = True
        params_list.extend(params_list_child)

    if not status and lr is None and params_list:  # 表明该模型中没有找到不同的学习率组件且没有指定的前继学习率
        # 同时，需要确保模型中的params_list非空(如果为空，则为所有子节点都被标记为不更新)
        return [{'params': model.parameters()}], status, update_mark

    return params_list, status, update_mark

In [3]:
import torch
from torch import nn
from torch import optim
import numpy as np
import pickle
from matplotlib import pyplot as plt
from matplotlib_inline import backend_inline


def init_weights(m):
    if type(m) == nn.Linear or type(m) == nn.Conv2d:
        nn.init.xavier_uniform_(m.weight)


def cpu():
    """Get the CPU device.

    Defined in :numref:`sec_use_gpu`"""
    return torch.device('cpu')


def gpu(i=0):
    """Get a GPU device.

    Defined in :numref:`sec_use_gpu`"""
    return torch.device(f'cuda:{i}')


def num_gpus():
    """Get the number of available GPUs.

    Defined in :numref:`sec_use_gpu`"""
    return torch.cuda.device_count()


def try_gpu(i=0):
    """Return gpu(i) if exists, otherwise return cpu().

    Defined in :numref:`sec_use_gpu`"""
    if num_gpus() >= i + 1:
        return gpu(i)
    return cpu()


class NNFrameWork(nn.Module):
    """
    通用神经网络训练框架
    核心框架结构梳理：

    >训练类方法:
    train_real_time(x, target) -> loss
    train_fixed(data_iter, epochs, [save_path, plot_hist, other_options]) -> loss <from last time>
        >训练函数接口(interface):
        _train_fixed_implementation
        >训练核心组件:
        _check_feasibility
        _get_gradient
        _self_update

    >模型衡量/应用方法:
    predict(with gradient/without gradient)
    evaluate_loss
    evaluate_accuracy

    >模型信息输入输出:
    plot_hist
    save_model
    load_model
    load_optimizer

    >模型基本架构设置:
    set_optimizer
    set_weights_init
    set_criterion
    set_device
    """

    def __init__(self):
        super().__init__()
        ################################
        self.optimizer = None
        self.criterion = None
        self.device = None
        self.clip_value = None
        self.init_fs = None
        #################################
        self.iter_times = 0  # input_num
        self.loss_history = []
        self.temp_loss_history = []
        self.to(self.device)

    def train_real_time(self, x, target):
        self._check_feasibility()
        self.train()

        # 更新队列及迭代次数
        self.iter_times += 1

        # 学习及更新
        loss = self._get_gradient(x, target)
        self._self_update()

        # 学习历史记录添加
        self.loss_history.append(loss)

    def train_fixed(self, data_iter, epochs, save_path=None, plot_hist=True,
                    other_options=None):
        self._check_feasibility()
        self.train()
        loss = self._train_fixed_implementation(data_iter, epochs, other_options)
        ###############################save and plot hist##############################
        if save_path is not None:
            torch.save(self.state_dict(), save_path + ".pt")
            torch.save(self.optimizer.state_dict(), save_path + '_optimizer_state.pth')
            with open(save_path + '_train_hist.pkl', 'wb') as f:
                pickle.dump(self.loss_history, f)
        if plot_hist:
            self.plot_hist()
        ###############################save and plot hist##############################
        return loss

    def _train_fixed_implementation(self, data_iter, epochs, other_options=None):

        loss_threshold = None  # loss阈值，用于判断迭代退出条件
        if other_options is not None:
            loss_threshold = other_options['loss_threshold']

        for epoch in range(epochs):
            for i, (x, y) in enumerate(data_iter):
                x = x.to(self.device)
                y = y.to(self.device)
                ##################training################
                # 学习及更新
                loss = self._get_gradient(x, y)
                if loss_threshold is not None:  # 判断loss是否已经达到要求
                    if loss < loss_threshold:
                        return loss
                self._self_update(x, y)
                # 学习历史记录添加
                if i == len(data_iter) - 1:
                    self.temp_loss_history.append(loss)
                    self.loss_history.append(np.mean(self.temp_loss_history))
                    self.temp_loss_history = []
                    # 更新队列及迭代次数
                    self.iter_times += 1
                else:
                    self.temp_loss_history.append(loss)
                ##################training################
        return self.loss_history[-1]

    def _check_feasibility(self):
        assert self.optimizer is not None, "No optimizer specified!"
        assert self.criterion is not None, "No criterion specified!"
        assert self.device is not None, "No device specified!"

    def _get_gradient(self, x, target):
        x = x.to(self.device)
        target = target.to(self.device)

        # 学习及更新
        output = self(x)
        loss = self.criterion(output, target)
        self.optimizer.zero_grad()
        loss.backward()
        return loss.item()

    def _self_update(self, data=None, target=None):
        if self.clip_value is not None:
            nn.utils.clip_grad_norm_(self.parameters(), self.clip_value)  # 梯度截断
        self.optimizer.step()

    def predict(self, x, no_grad=False):
        self.eval()
        if no_grad:
            with torch.no_grad():
                return self(x)
        return self(x)

    def evaluate_loss(self, data_iter):
        loss_list = []
        for i, (x, y) in enumerate(data_iter):
            x = x.to(self.device)
            y = y.to(self.device)
            ##################evaluate################
            with torch.no_grad():
                # 计算loss
                output = self.predict(x)
                loss = self.criterion(output, y)
                loss_list.append(loss.cpu())
            ##################evaluate################
        return np.mean(loss_list)

    def evaluate_accuracy(self, data_iter):
        self.eval()  # 将模型设置为评估模式
        correct = 0
        total = 0
        with torch.no_grad():  # 在评估模式下不需要计算梯度
            for data in data_iter:
                inputs, labels = data[0].to(self.device), data[1].to(self.device)  # 移动数据到设备（如GPU）
                outputs = self.predict(inputs, no_grad=True)
                _, predicted = torch.max(outputs, 1)  # 获取每行最大值的位置（即预测的类别）
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        accuracy = correct / total
        return accuracy

    def plot_hist(self):
        plt.figure()
        backend_inline.set_matplotlib_formats('svg')
        plt.plot(self.loss_history)
        plt.xlabel("epochs")
        plt.ylabel("loss")
        plt.show()

    def save_model(self, model_name="model"):
        torch.save(self.state_dict(), model_name + ".pt")
        torch.save(self.optimizer.state_dict(), model_name + '_optimizer_state.pth')
        with open(model_name + '_train_hist.pkl', 'wb') as f:
            pickle.dump(self.loss_history, f)

    def load_model(self, model_path_without_suffix):
        self.load_state_dict(torch.load(model_path_without_suffix + ".pt"))
        with open(model_path_without_suffix + '_train_hist.pkl', 'rb') as f:
            self.loss_history = pickle.load(f)

    def load_optimizer(self, model_path_without_suffix):
        self.optimizer.load_state_dict(torch.load(model_path_without_suffix + '_optimizer_state.pth'))

    def set_optimizer(self, lr=0.01, momentum=0.0, clip_value=float("inf"), optimizer="default"):
        self.clip_value = clip_value
        if optimizer == "default":
            self.optimizer = optim.SGD(self.get_param_groups(), lr=lr, momentum=momentum)
        else:
            self.optimizer = optimizer
        print(self.optimizer)

    def get_param_groups(self):
        params, _, _ = get_params_v2(self)
        return params

    def set_weights_init(self, init_f):
        self.init_fs.append(init_f)
        self.apply(init_f)

    def set_criterion(self, criterion):
        self.criterion = criterion

    def set_device(self, device=cpu()):
        self.device = device
        self.to(self.device)

In [4]:
class Sequential(nn.Sequential, NNFrameWork):
    pass

In [5]:
class A(NNFrameWork):
    def __init__(self):
        super().__init__()
        self.lr = 0.5
        self.update_mark = True

In [6]:
class C(NNFrameWork):
    def __init__(self):
        super().__init__()
        self.lr = 0.7
        self.update_mark = False

In [7]:
class B(NNFrameWork):
    def __init__(self):
        super().__init__()
        self.update_mark = True
        self.a = A()
        self.c = C()

In [8]:
get_params_v2(B())

([{'params': <generator object Module.parameters at 0x000002AE17B5F840>,
   'lr': 0.5}],
 True,
 True)

In [9]:
B().get_param_groups()

[{'params': <generator object Module.parameters at 0x000002AE17B5F8B0>,
  'lr': 0.5}]

In [10]:
# 定义一个装饰器函数，用于打印函数执行前后的时间
import time


def timer_decorator(func):
    def wrapper(*args, **kwargs):
        start_time = time.time()
        result = func(*args, **kwargs)
        end_time = time.time()
        print(f"{func.__name__} took {end_time - start_time:.4f} seconds to run.")
        return result

    return wrapper


# 使用装饰器
@timer_decorator
def example_function(n):
    """模拟一个耗时的任务"""
    time.sleep(n)
    print(f"Function finished execution after {n} seconds.")


# 调用函数
example_function(2)

Function finished execution after 2 seconds.
example_function took 2.0091 seconds to run.


In [11]:
def forward_switch(forward, model):
    def wrapper(*args, **kwargs):
        if hasattr(model, 'predict_on') and model.predict_on:  # 判断该节点是否需要更新参数
            return model.predict(*args, **kwargs)
        return model()

    return wrapper

In [12]:
# 定义一个装饰器
def my_decorator(method):
    def wrapper(self, *args, **kwargs):
        # 在这里可以访问self.value
        print(f"装饰器中访问的value={self.value}")
        # 调用原始方法
        method(self)

    return wrapper


class MyClass:
    def __init__(self, value):
        self.value = value

    @my_decorator
    def my_method(self):
        print(f"原始方法，value={self.value}")


# 创建类的实例并调用方法
obj = MyClass(10)
obj.my_method()

装饰器中访问的value=10
原始方法，value=10


In [62]:
def predict_switch(forward):
    def wrapper(self, *args, **kwargs):
        if hasattr(self, 'predict_mode') and self.predict_mode:  # 判断该节点是否需要更新参数
            print("no grad")
            self.eval()
            with torch.no_grad():
                return forward(self, *args, **kwargs)
        return forward(self, *args, **kwargs)

    return wrapper

In [63]:
class TestLinear(NNFrameWork):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(10, 5)
        self.predict_mode = True
        self.forward = predict_switch(self.forward)

    def forward(self, x):
        print("method in forward")
        return self.linear(x)

In [64]:
test = TestLinear()
X = torch.randn(5, 10)
test(X)

method in forward


tensor([[-0.2530, -0.2685,  0.5557, -0.2769, -0.8019],
        [-0.4478,  0.4212, -0.2734, -0.4236, -0.6014],
        [ 0.6597, -0.2140,  0.1578, -0.4315,  0.8780],
        [-0.1377,  1.2599, -0.0780,  1.2490,  0.3306],
        [-0.7025, -0.5401, -0.2875, -1.1068, -0.5077]],
       grad_fn=<AddmmBackward0>)

In [55]:
class A(NNFrameWork):
    def __init__(self):
        super().__init__()
        self.lr = 0.5
        self.update_mark = True


class C(NNFrameWork):
    def __init__(self):
        super().__init__()
        self.lr = 0.7
        self.update_mark = False


class B(NNFrameWork):
    def __init__(self):
        super().__init__()
        self.update_mark = True
        self.a = A()
        self.c = C()

In [56]:
a = A()
b = B()
c = C()
k = Sequential(a, b, c)
k.get_param_groups()

[{'params': <generator object Module.parameters at 0x000002AE1255E490>,
  'lr': 0.5},
 {'params': <generator object Module.parameters at 0x000002AE1255E5E0>,
  'lr': 0.5}]

In [52]:
c.update_mark = True
k.get_param_groups()

[{'params': <generator object Module.parameters at 0x000002AE1255D850>,
  'lr': 0.5},
 {'params': <generator object Module.parameters at 0x000002AE1255D9A0>,
  'lr': 0.5},
 {'params': <generator object Module.parameters at 0x000002AE1255D8C0>,
  'lr': 0.7}]

In [5]:
import torch
import torch.nn as nn


class ForwardSwitchMeta(type):
    def __new__(cls, name, bases, attrs):
        def predict_switch(func):
            def wrapper(self, *args, **kwargs):
                if hasattr(self, 'predict_mode') and self.predict_mode:
                    print("no grad")
                    self.eval()
                    with torch.no_grad():
                        return func(self, *args, **kwargs)
                return func(self, *args, **kwargs)
            return wrapper
        # 获取类的属性
        forward = attrs.get('forward', None)
        if forward is not None:
            # 如果类定义了 forward 方法，则应用装饰器
            attrs['forward'] = predict_switch(forward)
            print(f"{name} switch to prediction mode")
        return super().__new__(cls, name, bases, attrs)


class BaseLinear(nn.Module, metaclass=ForwardSwitchMeta):
    def __init__(self):
        super(BaseLinear, self).__init__()
        self.linear = nn.Linear(10, 5)
        self.predict_mode = True

    def forward(self, x):
        print("method in derived forward")
        return self.linear(x)


# 创建实例并调用
test = BaseLinear()
X = torch.randn(5, 10)
test(X)

BaseLinear switch to prediction mode
no grad
method in derived forward


tensor([[-1.2307, -0.7880, -1.0240, -0.7369,  0.9642],
        [-0.2169, -0.2904, -0.1030, -0.2500,  0.1783],
        [-0.2899, -0.0309, -0.1530, -0.0051,  0.2810],
        [-0.1931, -0.0553,  0.3336, -0.4479, -0.1722],
        [-0.9966, -0.2734, -1.3659, -1.7040,  1.0884]])

In [69]:
class CustomMeta(type):
    def __new__(cls, name, bases, attrs):
        # 打印类名
        print(f"Creating class named {name}")

        # 检查类是否有某个特定的方法
        if 'some_method' in attrs:
            print(f"The class {name} has a method named some_method.")

        # 修改类的属性
        attrs['modified_attribute'] = "This attribute was added by the metaclass."

        # 创建并返回新的类对象
        return super(CustomMeta, cls).__new__(cls, name, bases, attrs)


class MyClass(metaclass=CustomMeta):
    def some_method(self):
        print("This is some_method.")


# 创建 MyClass 的实例
instance = MyClass()
print(instance.modified_attribute)

Creating class named MyClass
The class MyClass has a method named some_method.
This attribute was added by the metaclass.


In [79]:
# 定义类方法
def class_method(cls):
    print(f"This is a class method of {cls.__name__}")


# 定义实例方法
def instance_method(self):
    print(f"This is an instance method of {self.__class__.__name__}")


# 使用 type() 创建类
MyClass = type(
    'MyClass',  # 类名
    (object,),  # 基类列表
    {
        'class_method': class_method,  # 类方法
        'instance_method': instance_method,  # 实例方法
        'class_attribute': 'I am a class attribute'  # 类属性
    }
)
MyClass.class_method = classmethod(MyClass.class_method)

# 调用类方法
MyClass.class_method()  # 输出: This is a class method of MyClass

# 创建类的实例
obj = MyClass()

# 调用实例方法
obj.instance_method()  # 输出: This is an instance method of MyClass

# 访问类属性
print(MyClass.class_attribute)  # 输出: I am a class attribute

This is a class method of MyClass
This is an instance method of MyClass
I am a class attribute


In [8]:
class T:
    def __init__(self):
        self.x = 0

t = T()
t.forward = lambda x: 10 * x

In [9]:
t.forward(100)

1000